In [6]:
# =====================
# 导入与全局配置
# =====================
import numpy as np
from scipy.io import wavfile
from scipy.fft import fft, ifft, rfft, irfft
import matplotlib.pyplot as plt
import os
from glob import glob

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False


def read_audio(path):
    """读取 wav 文件并归一化到 [-1, 1]
    返回: rate, signal (shape: [N, channels])
    """
    rate, signal = wavfile.read(path)
    signal = signal.astype(np.float64)

    if signal.ndim == 1:
        signal = signal[:, np.newaxis]

    # 逐声道归一化
    for ch in range(signal.shape[1]):
        ch_max = np.max(np.abs(signal[:, ch]))
        if ch_max > 0:
            signal[:, ch] /= ch_max

    return rate, signal

In [7]:
# =====================
# 降噪函数：余弦低通滤波
# =====================
def denoise(signal, rate, cutoff=3000, width=1000):
    """FFT 低通降噪，余弦锥形过渡带避免吉布斯振铃

    传递函数 H(f):
        H(f) = 1                                    ,  |f| <= fc
        H(f) = 0.5*(1+cos(pi*(|f|-fc)/W))           ,  fc < |f| <= fc+W
        H(f) = 0                                    ,  |f| > fc+W

    参数:
        signal: 输入 [N, channels], 值域 [-1, 1]
        rate:   采样率 (Hz)
        cutoff: 截止频率 fc (Hz)
        width:  过渡带宽 W (Hz)
    返回:
        denoised:  降噪后信号 [N, channels]
        H:         滤波器响应 (正频率, 用于绘图)
        freqs_pos: 正频率轴 (Hz)
    """
    N = signal.shape[0]
    X = rfft(signal, axis=0, n=N)
    freqs_pos = np.fft.rfftfreq(N, d=1.0 / rate)

    # 余弦锥形过渡带低通滤波器
    H = np.ones(len(freqs_pos), dtype=np.float64)
    in_transition = (freqs_pos > cutoff) & (freqs_pos <= cutoff + width)
    H[in_transition] = 0.5 * (
        1.0 + np.cos(np.pi * (freqs_pos[in_transition] - cutoff) / width)
    )
    H[freqs_pos > cutoff + width] = 0.0

    # 滤波
    Y = X * H[:, np.newaxis]
    denoised = irfft(Y, n=N, axis=0)

    return denoised, H, freqs_pos

In [8]:
# =====================
# 加密函数：频域置乱 + 相位旋转
# =====================
def encrypt(signal):
    """在频域对信号加密（复数置换 + 相位旋转）
    参数:
        signal: 输入信号 shape [N, channels]
    返回:
        encrypted:     加密后信号 shape [N, channels]
        permutation:   置换表（解密时需传入）
        phase_shift:   相位偏移（解密时需传入）
    """
    N, n_channels = signal.shape
    pos_end = N // 2 + 1
    # 可加密的频点 (跳过 DC，偶长度跳过 Nyquist)
    perm_range = np.arange(1, pos_end - (1 if N % 2 == 0 else 0))
    n_perm = len(perm_range)

    keys = []          # 每声道的 (permutation, phase_shift)
    encrypted_channels = []

    for ch in range(n_channels):
        X = rfft(signal[:, ch])

        np.random.seed(12345 + ch)
        permutation = np.random.permutation(n_perm)
        phase_shift = np.random.uniform(0, 2 * np.pi, n_perm)

        # 先置换复数值，再乘相位旋转因子
        X_enc = X.copy()
        X_enc[perm_range] = X_enc[perm_range][permutation]
        X_enc[perm_range] = X_enc[perm_range] * np.exp(1j * phase_shift)

        encrypted_channels.append(irfft(X_enc, n=N))
        keys.append((permutation, phase_shift))

    return np.column_stack(encrypted_channels), keys


# =====================
# 解密函数
# =====================
def decrypt(encrypted_signal, keys):
    """解密信号
    参数:
        encrypted_signal: 加密信号 shape [N, channels]
        keys:             encrypt() 返回的密钥列表
    返回:
        decrypted: 解密后信号 shape [N, channels]
    """
    N, n_channels = encrypted_signal.shape
    pos_end = N // 2 + 1
    perm_range = np.arange(1, pos_end - (1 if N % 2 == 0 else 0))
    n_perm = len(perm_range)

    decrypted_channels = []

    for ch in range(n_channels):
        permutation, phase_shift = keys[ch]
        X = rfft(encrypted_signal[:, ch])

        # 构造逆置换
        inv_perm = np.empty(n_perm, dtype=np.int64)
        inv_perm[permutation] = np.arange(n_perm)

        # 先除相位旋转，再逆置换
        X_dec = X.copy()
        X_dec[perm_range] = X_dec[perm_range] * np.exp(-1j * phase_shift)
        X_dec[perm_range] = X_dec[perm_range][inv_perm]

        decrypted_channels.append(irfft(X_dec, n=N))

    return np.column_stack(decrypted_channels)

In [9]:
# =====================
# 可视化辅助函数
# =====================
def plot_denoise(signal, denoised, rate, H, freqs_pos, fname, out_dir,
                 cutoff=3000, width=1000):
    """降噪对比图"""
    N = signal.shape[0]
    max_samples = min(N, int(30 * rate))
    t = np.arange(max_samples) / rate
    idx = 0

    s_orig = signal[:max_samples, idx]
    s_deno = denoised[:max_samples, idx]
    spec_orig = np.abs(rfft(s_orig))
    spec_deno = np.abs(rfft(s_deno))
    f_axis = np.fft.rfftfreq(max_samples, d=1.0 / rate)

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle(f'降噪处理 - {fname} (fc={cutoff}Hz, W={width}Hz)',
                 fontsize=14, fontweight='bold')

    # (0,0) 时域
    axes[0, 0].plot(t, s_orig, '#2166ac', lw=0.3, alpha=0.7, label='含噪信号')
    axes[0, 0].plot(t, s_deno, '#b2182b', lw=0.3, alpha=0.5, label='降噪后')
    axes[0, 0].set_title('时域对比（前30秒）')
    axes[0, 0].set_xlabel('时间 (s)'); axes[0, 0].set_ylabel('幅度')
    axes[0, 0].set_ylim(-1.1, 1.1)
    axes[0, 0].legend(loc='upper right', fontsize=8, framealpha=0.9)
    axes[0, 0].grid(True, alpha=0.3)

    # (0,1) 时域局部
    zoom = min(int(2 * rate), max_samples); t_z = t[:zoom]
    axes[0, 1].plot(t_z, s_orig[:zoom], '#2166ac', lw=0.8, alpha=0.7, label='含噪信号')
    axes[0, 1].plot(t_z, s_deno[:zoom], '#b2182b', lw=1.0, alpha=0.5, label='降噪后')
    axes[0, 1].set_title('时域对比（前2秒局部放大）')
    axes[0, 1].set_xlabel('时间 (s)'); axes[0, 1].set_ylabel('幅度')
    axes[0, 1].legend(loc='upper right', fontsize=8, framealpha=0.9)
    axes[0, 1].grid(True, alpha=0.3)

    # (1,0) 频谱对比
    axes[1, 0].semilogy(f_axis, spec_orig, '#2166ac', lw=0.6, alpha=0.7, label='含噪频谱')
    axes[1, 0].semilogy(f_axis, spec_deno, '#b2182b', lw=0.6, alpha=0.7, label='降噪后频谱')
    axes[1, 0].set_title('频谱对比（对数坐标）')
    axes[1, 0].set_xlabel('频率 (Hz)'); axes[1, 0].set_ylabel('幅度')
    axes[1, 0].set_xlim(0, rate / 2)
    axes[1, 0].legend(loc='upper right', fontsize=8, framealpha=0.9)
    axes[1, 0].grid(True, alpha=0.3)

    # (1,1) 滤波器频率响应
    xlim = min(cutoff + 2 * width, rate / 2)
    axes[1, 1].plot(freqs_pos, H, '#1a9641', lw=1.4)
    axes[1, 1].set_title('余弦过渡带低通滤波器响应')
    axes[1, 1].set_xlabel('频率 (Hz)'); axes[1, 1].set_ylabel('增益')
    axes[1, 1].set_xlim(0, xlim)
    axes[1, 1].set_ylim(-0.05, 1.1)
    axes[1, 1].axvline(cutoff, color='gray', ls='--', alpha=0.5, lw=0.8)
    axes[1, 1].axvline(cutoff + width, color='gray', ls='--', alpha=0.5, lw=0.8)
    axes[1, 1].annotate(f'fc={cutoff}Hz', xy=(cutoff, 0.55), fontsize=7,
                        color='gray', ha='right', rotation=90)
    axes[1, 1].annotate(f'fc+W={cutoff+width}Hz', xy=(cutoff+width, 0.55),
                        fontsize=7, color='gray', ha='left', rotation=90)
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, 'denoise_compare.png'), dpi=200)
    plt.savefig(os.path.join(out_dir, 'denoise_compare.pdf'))
    plt.close()


def plot_crypto(original, encrypted, decrypted, rate, error_max, fname, out_dir):
    """加密/解密对比图"""
    N = original.shape[0]
    max_samples = min(N, int(30 * rate))
    t = np.arange(max_samples) / rate; idx = 0
    a = original[:max_samples, idx]
    b = encrypted[:max_samples, idx]
    c = decrypted[:max_samples, idx]

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle(f'FFT 频域加密 / 解密 - {fname}', fontsize=14, fontweight='bold')

    for ax, sig, color, title in [
        (axes[0, 0], a, '#2166ac', '加密前信号（降噪后）'),
        (axes[0, 1], b, '#b2182b', '加密信号（时域类噪声）'),
        (axes[0, 2], c, '#1a9641', '解密信号')]:
        ax.plot(t, sig, color=color, lw=0.3)
        ax.set_title(title); ax.set_xlabel('时间 (s)'); ax.set_ylabel('幅度')
        ax.set_ylim(-1.1, 1.1); ax.grid(True, alpha=0.3)

    zoom = min(int(2 * rate), max_samples); t_z = t[:zoom]
    axes[1, 0].plot(t_z, a[:zoom], '#2166ac', lw=1.2, alpha=0.7, label='加密前')
    axes[1, 0].plot(t_z, c[:zoom], '#fdae61', lw=1.8, alpha=0.4, label='解密后', ls='--')
    axes[1, 0].set_title('加密前 vs 解密后（前2秒叠加）')
    axes[1, 0].set_xlabel('时间 (s)'); axes[1, 0].set_ylabel('幅度')
    axes[1, 0].legend(loc='upper right', fontsize=8, framealpha=0.9)
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].plot(t, np.abs(a - c), '#b2182b', lw=0.3)
    axes[1, 1].set_title(f'解密误差 (max = {error_max:.2e})')
    axes[1, 1].set_xlabel('时间 (s)'); axes[1, 1].set_ylabel('绝对误差')
    axes[1, 1].grid(True, alpha=0.3)

    f_ax = np.fft.rfftfreq(max_samples, d=1.0 / rate)
    axes[1, 2].semilogy(f_ax, np.abs(rfft(a)), '#2166ac', lw=0.5, alpha=0.7, label='加密前')
    axes[1, 2].semilogy(f_ax, np.abs(rfft(b)), '#b2182b', lw=0.4, alpha=0.5, label='加密后')
    axes[1, 2].set_title('频谱对比（对数坐标）')
    axes[1, 2].set_xlabel('频率 (Hz)'); axes[1, 2].set_ylabel('幅度')
    axes[1, 2].set_xlim(0, rate / 2)
    axes[1, 2].legend(loc='upper right', fontsize=8, framealpha=0.9)
    axes[1, 2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, 'encrypt_decrypt.png'), dpi=200)
    plt.savefig(os.path.join(out_dir, 'encrypt_decrypt.pdf'))
    plt.close()

In [10]:
# =====================
# 批量处理主程序
# =====================

wav_files = glob("data/*.wav")
if not wav_files:
    print("错误: data 文件夹中未找到 .wav 文件")
else:
    print(f"找到 {len(wav_files)} 个 wav 文件: {wav_files}\n")

    for wav_path in wav_files:
        fname = os.path.splitext(os.path.basename(wav_path))[0]
        out_dir = os.path.join("output", fname)
        os.makedirs(out_dir, exist_ok=True)

        print(f"{'='*50}")
        print(f"处理: {fname}")

        # 1. 读取
        rate, sig = read_audio(wav_path)
        print(f"  采样率={rate} Hz  点数={sig.shape[0]}  声道={sig.shape[1]}")

        # 2. 降噪 (cutoff=1500 Hz, width=1000 Hz → 阻带从2500Hz开始)
        denoised, H, freqs_pos = denoise(sig, rate, cutoff=1500)
        plot_denoise(sig, denoised, rate, H, freqs_pos, fname, out_dir,
                     cutoff=1500)
        wavfile.write(os.path.join(out_dir, "denoised.wav"),
                      rate, (denoised * 32767).astype(np.int16))
        print(f"  -> denoised.wav 已保存")

        # 3. 加密
        encrypted, keys = encrypt(denoised)

        # 4. 解密
        decrypted = decrypt(encrypted, keys)

        # 5. 验证
        err_max = np.max(np.abs(decrypted - denoised))
        err_rms = np.sqrt(np.mean((decrypted - denoised) ** 2))
        print(f"  解密误差: max={err_max:.2e}  rms={err_rms:.2e}")

        # 6. 可视化 & 保存
        plot_crypto(denoised, encrypted, decrypted, rate, err_max, fname, out_dir)
        wavfile.write(os.path.join(out_dir, "encrypted.wav"),
                      rate, (encrypted * 32767).astype(np.int16))
        wavfile.write(os.path.join(out_dir, "decrypted.wav"),
                      rate, (decrypted * 32767).astype(np.int16))
        print(f"  -> encrypted.wav / decrypted.wav 已保存\n")

    print(f"全部完成！共处理 {len(wav_files)} 个文件")

  解密误差: max=1.67e-15  rms=2.91e-16


Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], 

  -> encrypted.wav / decrypted.wav 已保存

全部完成！共处理 2 个文件
